# Alphamind新手入门之五：Alpha因子的五分位分析

In [ ]:
# %matplotlib inline
import asyncio
import datetime as dt


## 1. 数据准备
------------------

* 股票池：中证800
* 因子：`ETOP`
* 调仓频率：5个交易日
* 起始日：2010年1月1日
* 终止日：2019年1月31日
* 分位数：5

In [ ]:
# 因子中性化设置
# industry_stryles为行业风格   risk_styles 为风险风格
# makeSchedule函数 用于设置调仓日的生成工作日序列
import numpy as np
import pyarrow as pa


In [ ]:
#获取数据
#fetch_factor_range 获取因子数据
#fetch_risk_model_range 获取风险模型数据
#fetch_dx_return_range 获取收益率数据
#%%time
import pyarrow.compute as pc

rng = np.random.default_rng(42)


## 2. 分位数群组收益比较
-----------------------

In [ ]:
#neutralize 因子中性化函数
n_dates = 6
n_assets = 12


In [ ]:
# er_quantile_analysis 分位数分析函数
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.analysis.quantileanalysis import quantile_analysis


In [ ]:
latest = panel.filter(pc.equal(panel["trade_date"], dates[-1]))
returns = quantile_analysis(
    latest.select(["factor"]),
    np.array([1.0]),
    latest["dx"].to_numpy(zero_copy_only=False),
    n_bins=5,
    pre_process=[],
    post_process=[],
)


In [ ]:
assert returns.shape == (5,)
pa.table({"quantile": np.arange(1, 6), "return": returns})
